# Deep Learning QA: Model Instrumentation & Neuron Coverage Walkthrough

This walkthrough demonstrates how to build and use an **instrumentation model** (activation probe) in deep learning quality assurance.

### Objectives:
1. **Load and preprocess** the standard MNIST handwritten digit dataset.
2. **Train a baseline classifier** using `BasicNeuralNetwork`.
3. **Build an instrumentation model** using `InstrumentationModel` to tap intermediate activations without altering the original classifier.
4. **Compute Neuron Coverage** across test samples using normalized activation thresholds.
5. **Analyze coverage behavior** across varying activation thresholds and test suite sample sizes.

## 1. Imports and Environment Setup

Import necessary libraries and load `BasicNeuralNetwork` and `InstrumentationModel` from the `models` module.

In [3]:
import sys
from pathlib import Path
import numpy as np
import keras

# Ensure models module is accessible from parent directory
sys.path.append(str(Path.cwd().parent))

from models import BasicNeuralNetwork, InstrumentationModel

## 2. Load and Preprocess MNIST Dataset

Load the MNIST training and test sets, and scale pixel values to the range $[0.0, 1.0]$.

In [4]:
# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values from [0, 255] to [0.0, 1.0]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print(f"Training set: {x_train.shape}, Labels: {y_train.shape}")
print(f"Test set:     {x_test.shape}, Labels: {y_test.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Training set: (60000, 28, 28), Labels: (60000,)
Test set:     (10000, 28, 28), Labels: (10000,)


## 3. Instantiate and Compile `BasicNeuralNetwork`

Initialize the reusable basic feedforward model with input shape `(28, 28)` and 10 output classes.

In [5]:
# Create and compile model
model = BasicNeuralNetwork(input_shape=(28, 28), num_classes=10, name="mnist_classifier")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

model.summary()

Model: "mnist_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_64 (Dense)                │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,650 (205.66 KB)

 Trainable params: 52,650 (205.66 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Train the Model

Fit the model on the training data.

In [6]:
# Train baseline model
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8715 - loss: 0.4602 - val_accuracy: 0.9490 - val_loss: 0.1923
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9418 - loss: 0.1988 - val_accuracy: 0.9598 - val_loss: 0.1443
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 904us/step - accuracy: 0.9569 - loss: 0.1465 - val_accuracy: 0.9672 - val_loss: 0.1228
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 935us/step - accuracy: 0.9641 - loss: 0.1185 - val_accuracy: 0.9700 - val_loss: 0.1085
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9706 - loss: 0.0989 - val_accuracy: 0.9728 - val_loss: 0.0964


## 5. Evaluate the Classifier

Evaluate test loss and accuracy on unseen test samples.

In [7]:
# Evaluate test accuracy
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Test Loss:     0.1115
Test Accuracy: 96.51%


## 6. Build the Instrumentation Model (Activation Probe)

Initialize `InstrumentationModel` wrapping the base model. This creates a functional probe exposing the intermediate activations of the hidden `Dense` layers (excluding the final `predictions` output layer).

In [8]:
# Wrap the model with our instrumentation framework
activation_model = InstrumentationModel(model, name="activation_probe")

print(f"Instrumentation Model Name: {activation_model.name}")
print(f"Number of Probed Hidden Layers: {len(activation_model.hidden_layers)}")
for i, layer in enumerate(activation_model.hidden_layers):
    print(f"  Layer {i+1}: {layer.name} ({layer.units} neurons, activation={layer.activation.__name__})")

Instrumentation Model Name: activation_probe
Number of Probed Hidden Layers: 2
  Layer 1: dense_64 (64 neurons, activation=relu)
  Layer 2: dense_32 (32 neurons, activation=relu)


## 7. Define Normalization and Neuron Coverage Functions

Implement `normalize` and `neuron_coverage` to scale activations per neuron to $[0, 1]$ and compute the ratio of neurons activated above a given threshold across input samples.

In [9]:
def normalize(a, lo, hi):
    """
    Normalize activations per neuron to [0, 1] range.
    Safely handles constant activations where max == min to prevent division by zero.
    """
    diff = hi - lo
    diff = np.where(diff == 0, 1.0, diff)
    return (a - lo) / diff


def neuron_coverage(xs, threshold=0):
    """
    Calculate neuron coverage for test set xs across all probed hidden layers.
    """
    acts = activation_model.predict(xs, verbose=0)

    mins = [a.min(axis=0) for a in acts]
    maxs = [a.max(axis=0) for a in acts]

    covered_per_layer = []

    for a, lo, hi in zip(acts, mins, maxs):
        scaled = normalize(a, lo, hi)
        covered = np.any(scaled > threshold, axis=0)
        covered_per_layer.append(covered.ravel())

    flags = np.concatenate(covered_per_layer)

    return flags.mean()

## 8. Calculate Neuron Coverage Across Different Thresholds

Measure neuron coverage across a range of activation thresholds $t \in [0.0, 0.2, 0.4, 0.6, 0.8]$.

In [10]:
# Measure coverage on a test subset across activation thresholds
test_subset = x_test[:1000]
thresholds = [0.0, 0.2, 0.4, 0.6, 0.8]

print(f"Neuron Coverage for {len(test_subset)} test samples:")
print("-" * 45)
for t in thresholds:
    cov = neuron_coverage(test_subset, threshold=t)
    print(f"Threshold = {t:.1f}  ->  Neuron Coverage: {cov * 100:6.2f}%")

Neuron Coverage for 1000 test samples:
---------------------------------------------
Threshold = 0.0  ->  Neuron Coverage:  98.96%
Threshold = 0.2  ->  Neuron Coverage:  98.96%
Threshold = 0.4  ->  Neuron Coverage:  98.96%
Threshold = 0.6  ->  Neuron Coverage:  98.96%
Threshold = 0.8  ->  Neuron Coverage:  98.96%


## 9. Analyze Coverage Growth vs. Test Suite Size

Evaluate how neuron coverage increases as test suite size expands from small subsets to thousands of test samples.

In [11]:
# Analyze coverage scaling with test set size
sample_sizes = [5, 20, 50, 100, 500, 1000, 5000]
t_eval = 0.5

print(f"Coverage Scaling (Threshold = {t_eval}):")
print("-" * 45)
for size in sample_sizes:
    subset = x_test[:size]
    cov = neuron_coverage(subset, threshold=t_eval)
    print(f"Test Size: {size:5d}  ->  Neuron Coverage: {cov * 100:6.2f}%")

Coverage Scaling (Threshold = 0.5):
---------------------------------------------
Test Size:     5  ->  Neuron Coverage:  96.88%
Test Size:    20  ->  Neuron Coverage:  98.96%
Test Size:    50  ->  Neuron Coverage:  98.96%
Test Size:   100  ->  Neuron Coverage:  98.96%
Test Size:   500  ->  Neuron Coverage:  98.96%
Test Size:  1000  ->  Neuron Coverage:  98.96%
Test Size:  5000  ->  Neuron Coverage:  98.96%
